In [ ]:
/**
 * ============================================================
 * SINGLE POINT OF FAILURE (SPOF) — SYSTEM DESIGN
 * ============================================================
 *
 * Core Idea:
 *
 *     SPOF = Single Point of Failure
 *
 *     Aisa component/dependency/process jiske fail hone
 *     par poora system ya koi CRITICAL user flow fail
 *     ho jaye.
 *
 *
 * IMPORTANT:
 *
 * SPOF sirf server nahi hota.
 *
 * SPOF ho sakta hai:
 *
 *     Database
 *     Load Balancer
 *     DNS
 *     Cloud Region
 *     Message Broker
 *     Secrets Manager
 *     Deployment Pipeline
 *     Feature Flag Service
 *     Network Path
 *     Human/Expert
 *
 *
 * Example:
 *
 *     Users
 *        |
 *        v
 *     Load Balancer
 *        |
 *      /   \
 *     v     v
 *    App1  App2
 *
 *
 *     App1/App2 healthy hain
 *
 *     BUT
 *
 *     Load Balancer fails
 *            ↓
 *     Users cannot reach App1/App2
 *
 *
 *     Therefore:
 *
 *     Load Balancer = SPOF
 *
 *
 * ============================================================
 * 1. WHAT MAKES SOMETHING A SPOF?
 * ============================================================
 *
 * Kisi component ko SPOF bolne ke liye
 * 3 conditions important hain:
 *
 *
 *     1. Critical path depends on it
 *
 *     2. Failure ke time koi working alternative nahi hai
 *
 *     3. Failure ka impact unacceptable hai
 *
 *
 * Simple formula:
 *
 *
 *     Critical Dependency
 *            +
 *     No Working Alternative
 *            +
 *     High Impact
 *            =
 *           SPOF
 *
 *
 * ============================================================
 * 2. NOT EVERY FAILURE IS A SPOF
 * ============================================================
 *
 * Important:
 *
 *     Har failed component SPOF nahi hota.
 *
 *
 * Example:
 *
 *     Recommendation Service
 *              |
 *             FAIL
 *
 * Homepage:
 *
 *     Popular products dikha sakti hai.
 *
 *
 * Therefore:
 *
 *     Checkout -> still works
 *     Homepage -> still works
 *
 *
 * Recommendation service:
 *
 *     SPOF for personalized recommendations
 *     BUT
 *     NOT SPOF for entire product.
 *
 *
 * ============================================================
 * 3. SPOF IS ABOUT USER FLOWS
 * ============================================================
 *
 * SPOF analysis ka focus:
 *
 *     "Kitne servers hain?"
 *
 * nahi.
 *
 * Focus:
 *
 *     "Agar ye dependency fail hui,
 *      user ka kya impact hoga?"
 *
 *
 * Therefore:
 *
 *     SPOF analysis =
 *     User Flows + Dependencies + Failure Impact
 *
 *
 * ============================================================
 * 4. EXAMPLE ARCHITECTURE
 * ============================================================
 *
 *
 *     Clients
 *         |
 *         v
 *     Load Balancer
 *       (SPOF)
 *         |
 *       /   \
 *      v     v
 *    App1   App2
 *      \     /
 *       \   /
 *        Cache
 *      (Cascading Risk)
 *          |
 *          v
 *       Database
 *         (SPOF)
 *
 *
 * Load Balancer:
 *
 *     If it fails:
 *
 *         Clients
 *             X
 *         App servers
 *
 *     Users cannot reach application.
 *
 *
 * Database:
 *
 *     If it fails:
 *
 *     Reads/writes requiring durable data
 *     stop.
 *
 *
 * Cache:
 *
 *     Cache failure may NOT directly stop
 *     the application.
 *
 *     BUT:
 *
 *     Cache fails
 *          ↓
 *     Requests go to DB
 *          ↓
 *     DB receives huge traffic
 *          ↓
 *     DB overload
 *          ↓
 *     DB fails
 *
 *
 * This is called:
 *
 *     CASCADING FAILURE
 *
 *
 * ============================================================
 * 5. HIDDEN SPOFs
 * ============================================================
 *
 * SPOF architecture diagram mein obvious nahi hota.
 *
 *
 * Example:
 *
 *     App Server 1
 *     App Server 2
 *
 * Looks redundant.
 *
 * But both depend on:
 *
 *     One NAT Gateway
 *
 *
 * NAT Gateway fails:
 *
 *     App1 -> Network unavailable
 *     App2 -> Network unavailable
 *
 *
 * Therefore:
 *
 *     NAT Gateway = Hidden SPOF
 *
 *
 * Other hidden SPOFs:
 *
 *     One firewall
 *     One route
 *     One private network link
 *     One DNS
 *     One control-plane service
 *
 *
 * ============================================================
 * 6. REDUNDANCY ON PAPER vs REAL REDUNDANCY
 * ============================================================
 *
 * Suppose:
 *
 *     App Server 1
 *     App Server 2
 *
 * Dono available hain.
 *
 * But each server can handle:
 *
 *     50% traffic
 *
 * Total traffic:
 *
 *     100%
 *
 *
 * Server 1 fails:
 *
 *     Server 2 gets 100%
 *
 * But Server 2 can handle only 50%.
 *
 *
 * Result:
 *
 *     Server 2 overloads
 *          ↓
 *     System fails
 *
 *
 * Therefore:
 *
 *     Two servers != automatically redundant
 *
 *
 * Real redundancy requires:
 *
 *     Capacity
 *     Health checks
 *     Proper routing
 *     Failure isolation
 *
 *
 * ============================================================
 * 7. COMMON SPOFs
 * ============================================================
 *
 *
 * TRAFFIC ENTRY
 *
 *     SPOF:
 *         One Load Balancer
 *         One Ingress
 *         One DNS provider
 *
 *     Impact:
 *         Users cannot reach service.
 *
 *
 * COMPUTE
 *
 *     SPOF:
 *         One application instance
 *         One Availability Zone
 *
 *     Impact:
 *         Requests fail when it goes down.
 *
 *
 * DATA
 *
 *     SPOF:
 *         One database primary
 *         without failover
 *
 *     Impact:
 *         Reads/writes stop.
 *
 *
 * CACHE
 *
 *     SPOF:
 *         One cache cluster
 *         without proper fallback
 *
 *     Impact:
 *         Database overload after cache failure.
 *
 *
 * MESSAGING
 *
 *     SPOF:
 *         One message broker/queue/stream
 *
 *     Impact:
 *         Producers block
 *         Consumers stop
 *
 *
 * CONFIGURATION
 *
 *     SPOF:
 *         One config service
 *         One feature flag service
 *
 *     Impact:
 *         Services cannot safely start/update.
 *
 *
 * SECRETS
 *
 *     SPOF:
 *         One secrets manager
 *         One token issuer
 *
 *     Impact:
 *         Authentication/startup failure.
 *
 *
 * STORAGE
 *
 *     SPOF:
 *         One bucket
 *         One volume
 *         One filesystem
 *
 *     Impact:
 *         Files become unavailable.
 *
 *
 * NETWORK
 *
 *     SPOF:
 *         One NAT gateway
 *         One route table
 *         One firewall
 *         One VPN
 *         One private link
 *
 *     Impact:
 *         Large part of system loses connectivity.
 *
 *
 * OPERATIONS
 *
 *     SPOF:
 *         One deployment pipeline
 *         One runbook
 *         One cloud account
 *         One expert
 *
 *     Impact:
 *         Recovery depends on one process/person.
 *
 *
 * ============================================================
 * 8. OPERATIONAL SPOF
 * ============================================================
 *
 * SPOF sirf runtime architecture mein nahi hota.
 *
 * Incident ke time operational dependencies
 * bhi SPOF ban sakti hain.
 *
 *
 * Example:
 *
 *     Database has multiple replicas.
 *
 *     BUT
 *
 *     Failover manually karna padta hai.
 *
 *     Aur sirf ek engineer ko pata hai kaise karna hai.
 *
 *
 * Engineer unavailable:
 *
 *     Failover cannot happen.
 *
 *
 * Therefore:
 *
 *     Human knowledge = SPOF
 *
 *
 * Another example:
 *
 *     Backup exists
 *          ↓
 *     Restore permission
 *          ↓
 *     One unavailable account
 *
 *
 * Backup exists,
 * but restore nahi kar sakte.
 *
 *
 * ============================================================
 * 9. AI / ML SYSTEMS MEIN SPOF
 * ============================================================
 *
 * AI systems mein bhi SPOF ho sakte hain.
 *
 *
 * Examples:
 *
 *     One Vector Database
 *     One Model Gateway
 *     One Queue
 *     One Embedding Service
 *     One Policy Service
 *     One Feature Store
 *     One Quota Service
 *
 *
 * Example:
 *
 *     Application
 *         |
 *         v
 *     Model Gateway
 *         |
 *         v
 *     LLM
 *
 *
 * Model healthy hai.
 *
 * But:
 *
 *     Model Gateway fails
 *
 * Then:
 *
 *     Application -> LLM
 *
 *     Cannot reach LLM.
 *
 *
 * Therefore:
 *
 *     Model Gateway = SPOF
 *
 *
 * ============================================================
 * 10. FAIL-CLOSED DECISION
 * ============================================================
 *
 * Kuch services intentionally fail-closed
 * kar sakti hain.
 *
 *
 * Example:
 *
 *     Policy Service
 *           |
 *         FAIL
 *           ↓
 *     High-risk request
 *
 * Instead of allowing request:
 *
 *     REJECT
 *
 *
 * Security reasons ke liye ye correct behavior
 * ho sakta hai.
 *
 *
 * Important:
 *
 *     Fail-closed behavior intentional,
 *     documented aur understood hona chahiye.
 *
 *     Accidental failure nahi hona chahiye.
 *
 *
 * ============================================================
 * 11. HOW TO IDENTIFY SPOFs
 * ============================================================
 *
 * SPOF analysis basically:
 *
 *     Dependency Analysis
 *
 *
 * Step 1:
 *
 *     Critical user flows identify karo.
 *
 *
 * Examples:
 *
 *     Login
 *     Checkout
 *     Payment
 *     Search
 *     Document Upload
 *     Model Inference
 *     Admin Permission Change
 *     Incident Rollback
 *
 *
 * Step 2:
 *
 *     Har flow ki dependencies list karo.
 *
 *
 * Include:
 *
 *     Services
 *     Databases
 *     Caches
 *     Queues
 *     Secrets
 *     DNS
 *     Network
 *     Third-party APIs
 *     Control-plane services
 *
 *
 * Step 3:
 *
 *     Har dependency ke liye ask:
 *
 *     "Agar ye fail ho gaya,
 *      user flow ka kya hoga?"
 *
 *
 * ============================================================
 * 12. RUNTIME vs RECOVERY DEPENDENCY
 * ============================================================
 *
 * System ke generally 2 dependency graphs hote hain:
 *
 *
 * 1. RUNTIME GRAPH
 *
 *     Normal traffic serve karne ke liye
 *     kya-kya required hai?
 *
 *
 * 2. RECOVERY GRAPH
 *
 *     Failure ke baad system ko repair karne ke liye
 *     kya-kya required hai?
 *
 *
 * ============================================================
 * RUNTIME EXAMPLE
 * ============================================================
 *
 *     Client
 *       |
 *       v
 *     API
 *       |
 *       v
 *     Database
 *
 *
 * Runtime dependency:
 *
 *     Database
 *
 *
 * ============================================================
 * RECOVERY EXAMPLE
 * ============================================================
 *
 * Database has replicas.
 *
 * But failover ke liye:
 *
 *     Secrets Manager
 *          +
 *     Deployment Tool
 *          +
 *     DNS
 *          +
 *     Manual Permission
 *
 *
 * required hain.
 *
 *
 * Agar Secrets Manager down hai:
 *
 *     Application maybe still serving traffic
 *
 *     BUT
 *
 *     Restart/recovery impossible ho sakta hai.
 *
 *
 * Therefore:
 *
 *     Recovery dependency bhi SPOF ban sakti hai.
 *
 *
 * ============================================================
 * 13. SHARED FATE
 * ============================================================
 *
 * Important concept:
 *
 *     Do components tabhi truly redundant hain
 *     jab woh independently fail kar sakein.
 *
 *
 * Example:
 *
 *     Primary DB
 *         |
 *     Replica DB
 *
 * Dono same Availability Zone mein hain.
 *
 *
 * AZ fails:
 *
 *     Primary -> FAIL
 *     Replica -> FAIL
 *
 *
 * So:
 *
 *     Replica exists
 *     BUT
 *     real redundancy nahi hai.
 *
 *
 * This is:
 *
 *     SHARED FATE
 *
 *
 * ============================================================
 * 14. SHARED FATE EXAMPLES
 * ============================================================
 *
 *
 * Example 1:
 *
 *     Primary DB
 *     Replica DB
 *
 *     Same AZ
 *
 *     AZ failure
 *          ↓
 *     Both fail
 *
 *
 * Example 2:
 *
 *     Load Balancer A
 *     Load Balancer B
 *
 *     Both depend on:
 *
 *         One DNS
 *
 *     DNS fails
 *          ↓
 *     Both unreachable
 *
 *
 * Example 3:
 *
 *     Two services
 *          |
 *          v
 *     Same Database
 *
 *     Database fails
 *          ↓
 *     Both services affected
 *
 *
 * Example 4:
 *
 *     Primary data
 *     Backup
 *
 *     Same cloud account
 *
 *     Account compromised/deleted
 *          ↓
 *     Both data + backup affected
 *
 *
 * KEY IDEA:
 *
 *     Redundancy is useful only when
 *     failure modes are sufficiently independent.
 *
 *
 * ============================================================
 * 15. FAILURE REVIEW
 * ============================================================
 *
 * Har important component ke liye ask:
 *
 *
 *     1. What if it becomes slow?
 *
 *     2. What if it returns errors?
 *
 *     3. What if it returns bad data?
 *
 *     4. What if it is unreachable from
 *        one zone/region?
 *
 *     5. What if it recovers with stale data?
 *
 *     6. What if failover works but
 *        capacity is reduced?
 *
 *
 * IMPORTANT:
 *
 * Slow dependency is especially dangerous.
 *
 *
 * Hard failure:
 *
 *     Health check detects failure
 *
 *
 * Slow dependency:
 *
 *     Threads wait
 *     Connection pools fill
 *     Queues grow
 *     Retries increase
 *
 *          ↓
 *
 *     Entire service can degrade.
 *
 *
 * ============================================================
 * 16. TEST YOUR SPOF ASSUMPTIONS
 * ============================================================
 *
 * Architecture diagram sirf intended behavior
 * dikhata hai.
 *
 * Real reliability check ke liye:
 *
 *     Controlled failure testing
 *
 * karo.
 *
 *
 * Start small:
 *
 *     Kill one app instance
 *     Disable one cache node
 *     Block database replica
 *     Rebalance queue consumers
 *
 *
 * Then bigger tests:
 *
 *     Simulate AZ failure
 *     Break DNS in test environment
 *     Test backup restore
 *     Failover to another region
 *
 *
 * ============================================================
 * 17. CHAOS ENGINEERING
 * ============================================================
 *
 * Chaos engineering:
 *
 *     Intentionally failures create karke
 *     system ka behavior test karna.
 *
 *
 * BUT:
 *
 * Random failure injection alone is not enough.
 *
 * Better approach:
 *
 *     Define hypothesis first.
 *
 *
 * Example:
 *
 *     "If Cache Cluster A fails:
 *
 *       Checkout latency < 300 ms
 *
 *       AND
 *
 *       Database CPU < 70%"
 *
 *
 * Then failure inject karo
 * and check whether hypothesis true hai.
 *
 *
 * ============================================================
 * 18. STRATEGIES TO REDUCE SPOFs
 * ============================================================
 *
 * Main strategies:
 *
 *     1. Redundancy
 *     2. Load balancing
 *     3. Health checks
 *     4. Data replication
 *     5. Backup
 *     6. Failure-domain isolation
 *     7. Graceful degradation
 *     8. Timeouts
 *     9. Retries with jitter
 *     10. Circuit breakers
 *     11. Backpressure
 *     12. Monitoring
 *     13. Runbooks
 *
 *
 * ============================================================
 * 19. REDUNDANCY
 * ============================================================
 *
 * Redundancy means:
 *
 *     Backup component available
 *     to take over after failure.
 *
 *
 * Examples:
 *
 *     Multiple app instances
 *     DB replicas
 *     Multiple cache nodes
 *     Multiple message brokers
 *     Multiple DNS nameservers
 *     Multiple AZs
 *     Multiple trained engineers
 *
 *
 * Two common models:
 *
 *
 * ACTIVE-ACTIVE
 *
 *     Multiple components
 *     simultaneously serve traffic.
 *
 *
 *     User
 *      |
 *      v
 *     LB
 *     / \
 *    v   v
 *   S1   S2
 *
 *
 * Benefits:
 *
 *     Better capacity
 *     Faster failover
 *
 *
 * ============================================================
 * ACTIVE-PASSIVE
 * ============================================================
 *
 * One component handles traffic.
 *
 * Another waits as standby.
 *
 *
 *     Primary
 *       |
 *     Traffic
 *
 *     Standby
 *       |
 *     Waiting
 *
 *
 * Primary fails:
 *
 *     Standby becomes Primary.
 *
 *
 * Benefits:
 *
 *     Simpler to reason about.
 *
 *
 * Risk:
 *
 *     Failover path may not work
 *     if never tested.
 *
 *
 * IMPORTANT:
 *
 *     Standby that has never been tested
 *     cannot be fully trusted.
 *
 *
 * ============================================================
 * 20. LOAD BALANCING + HEALTH CHECKS
 * ============================================================
 *
 * Load Balancer:
 *
 *     Traffic
 *        |
 *        v
 *     Load Balancer
 *       / | \
 *      v  v  v
 *     S1 S2 S3
 *
 *
 * Health checks determine:
 *
 *     Which instances are actually healthy?
 *
 *
 * IMPORTANT:
 *
 * Health check should check:
 *
 *     READINESS
 *
 * not just:
 *
 *     PROCESS IS ALIVE
 *
 *
 * Example:
 *
 *     Process running
 *          ↓
 *     Database unavailable
 *
 * Process:
 *
 *     Alive = YES
 *     Ready = NO
 *
 *
 * Traffic should not be sent to it.
 *
 *
 * ============================================================
 * 21. REQUEST DRAINING
 * ============================================================
 *
 * Failed/unhealthy server ko immediately
 * kill karne ke bajay possible ho to:
 *
 *     Existing requests complete hone do.
 *
 *
 * New requests:
 *
 *     Do not send.
 *
 *
 * Existing requests:
 *
 *     Drain
 *
 *
 * This avoids unnecessary request failures.
 *
 *
 * ============================================================
 * 22. LOAD BALANCER ITSELF CAN BE SPOF
 * ============================================================
 *
 * Bad design:
 *
 *
 *     Clients
 *        |
 *        v
 *     LB
 *      |
 *    /   \
 *   v     v
 *  S1    S2
 *
 *
 * LB fails:
 *
 *     Everything becomes unreachable.
 *
 *
 * Better:
 *
 *
 *     Clients
 *        |
 *     DNS / Anycast / VIP
 *       /       \
 *      v         v
 *     LB-A      LB-B
 *      |         |
 *     S1/S2     S3
 *
 *
 * Now:
 *
 *     LB-A fails
 *
 *     LB-B can continue serving traffic.
 *
 *
 * ============================================================
 * 23. DATA REPLICATION
 * ============================================================
 *
 * Replication:
 *
 *     Data ki multiple copies maintain karna.
 *
 *
 * Example:
 *
 *     Primary DB
 *       /     \
 *      v       v
 *   Replica A Replica B
 *
 *
 * Primary fails:
 *
 *     Another replica can potentially
 *     take over.
 *
 *
 * ============================================================
 * 24. SYNCHRONOUS REPLICATION
 * ============================================================
 *
 * Write:
 *
 *     Primary
 *        |
 *        v
 *     Replica
 *
 * Commit tab complete hota hai
 * jab replica/quorum acknowledge kare.
 *
 *
 * Advantages:
 *
 *     Better freshness
 *     Less replication lag
 *
 *
 * Disadvantage:
 *
 *     Higher latency
 *
 *
 * ============================================================
 * 25. ASYNCHRONOUS REPLICATION
 * ============================================================
 *
 * Write:
 *
 *     Client
 *       |
 *       v
 *     Primary
 *       |
 *     ACK immediately
 *       |
 *       v
 *     Replica catches up later
 *
 *
 * Advantages:
 *
 *     Lower latency
 *
 *
 * Disadvantage:
 *
 *     Failover ke time recent data
 *     lost ho sakta hai.
 *
 *
 * ============================================================
 * 26. MULTI-PRIMARY REPLICATION
 * ============================================================
 *
 * Multiple database nodes
 * writes accept kar sakte hain.
 *
 *
 * Benefits:
 *
 *     Higher availability
 *
 *
 * Problem:
 *
 *     Write conflicts.
 *
 *
 * Therefore:
 *
 *     Conflict resolution required.
 *
 *
 * ============================================================
 * 27. READ REPLICAS
 * ============================================================
 *
 * Read traffic distribute karne ke liye:
 *
 *
 *     Primary DB
 *        |
 *      Replicate
 *      /      \
 *     v        v
 *  Read      Read
 * Replica A Replica B
 *
 *
 * Benefits:
 *
 *     Better read capacity
 *     Additional failover option
 *
 *
 * Problem:
 *
 *     Replicas may lag behind primary.
 *
 *
 * ============================================================
 * 28. REPLICATION != BACKUP
 * ============================================================
 *
 * VERY IMPORTANT INTERVIEW QUESTION:
 *
 *     "Is replication same as backup?"
 *
 * Answer:
 *
 *     NO.
 *
 *
 * Replication:
 *
 *     Keeps another copy updated.
 *
 *
 * Problem:
 *
 *     Bad data also replicate ho sakta hai.
 *
 *
 * Example:
 *
 *     Application bug
 *          ↓
 *     Deletes 1 million records
 *          ↓
 *     Replication
 *          ↓
 *     Delete spreads to replicas
 *
 *
 * Backup:
 *
 *     Protects against:
 *
 *     Accidental deletion
 *     Bad application code
 *     Operator mistake
 *     Data corruption
 *     Ransomware
 *
 *
 * Therefore:
 *
 *     Critical data:
 *
 *         Replication + Backup
 *
 *
 * ============================================================
 * 29. BACKUP MUST BE TESTED
 * ============================================================
 *
 * Backup exists ≠ Recovery guaranteed.
 *
 *
 * Test:
 *
 *     Restore time
 *     Restore correctness
 *     Restore permissions
 *
 *
 * Important:
 *
 *     Backup ko at least once
 *     end-to-end restore karke test karo.
 *
 *
 * Otherwise:
 *
 *     Backup recoverability = unproven.
 *
 *
 * ============================================================
 * 30. GEOGRAPHIC / FAILURE-DOMAIN ISOLATION
 * ============================================================
 *
 * Components ko independent failure domains
 * mein distribute karo.
 *
 *
 * Possible domains:
 *
 *     Process
 *     Host
 *     Rack
 *     Availability Zone
 *     Region
 *     Cloud Account
 *     Cloud Provider
 *
 *
 * Example:
 *
 *     Bad:
 *
 *     Primary DB
 *     Replica DB
 *
 *     Same AZ
 *
 *
 * Better:
 *
 *     Primary DB -> AZ-A
 *
 *     Replica DB -> AZ-B
 *
 *
 * Now:
 *
 *     AZ-A failure
 *          ↓
 *     Replica in AZ-B survives.
 *
 *
 * ============================================================
 * 31. MULTI-AZ vs MULTI-REGION
 * ============================================================
 *
 *
 * MULTI-AZ
 *
 *     Usually a good first step
 *     for high availability.
 *
 *
 * MULTI-REGION
 *
 *     Useful when system needs:
 *
 *     Regional disaster recovery
 *     Global low latency
 *
 *
 * But multi-region introduces:
 *
 *     Replication lag
 *     Consistency trade-offs
 *     Failover complexity
 *     Higher operational cost
 *
 *
 * IMPORTANT:
 *
 * Don't use multi-region just because
 * "more redundancy is better."
 *
 * Business requirement dekho.
 *
 *
 * ============================================================
 * 32. GRACEFUL DEGRADATION
 * ============================================================
 *
 * Goal:
 *
 *     Optional dependency fail ho,
 *     but core product continues.
 *
 *
 * Example:
 *
 *     Recommendation Service
 *            |
 *           FAIL
 *            ↓
 *     Show trending products
 *
 *
 * Core transaction:
 *
 *     STILL WORKS
 *
 *
 * ============================================================
 * 33. GRACEFUL DEGRADATION EXAMPLES
 * ============================================================
 *
 *
 * Recommendations fail:
 *
 *     Personalized recommendations
 *            ↓
 *     Trending items
 *
 *
 * Search degraded:
 *
 *     Full search
 *         ↓
 *     Exact title matching
 *         ↓
 *     Cached results
 *
 *
 * Analytics delayed:
 *
 *     Show last-updated timestamp
 *
 *
 * Third-party enrichment fails:
 *
 *     Continue core transaction
 *     without enrichment.
 *
 *
 * Vector index rebuilding:
 *
 *     Don't return confusing empty results.
 *
 *     Show:
 *
 *         "Ingestion/index is being rebuilt."
 *
 *
 * IMPORTANT:
 *
 * Explicit limited-mode response
 * is better than silently returning
 * stale/incomplete data.
 *
 *
 * ============================================================
 * 34. TIMEOUTS
 * ============================================================
 *
 * Remote service ko indefinitely wait
 * nahi karna chahiye.
 *
 *
 * Example:
 *
 *     Service A
 *        |
 *        v
 *     Service B
 *
 * B slow hai.
 *
 * Without timeout:
 *
 *     A keeps waiting
 *        ↓
 *     Threads occupied
 *        ↓
 *     Connection pool full
 *        ↓
 *     A becomes unhealthy
 *
 *
 * Therefore:
 *
 *     Every remote call should have
 *     appropriate timeout.
 *
 *
 * ============================================================
 * 35. RETRIES + JITTER
 * ============================================================
 *
 * Temporary failure mein retry useful ho sakta hai.
 *
 * BUT:
 *
 * Too many retries:
 *
 *     Failure
 *       ↓
 *     Retry
 *       ↓
 *     More traffic
 *       ↓
 *     More failure
 *
 *
 * Therefore:
 *
 *     Use bounded retries.
 *
 *
 * Add:
 *
 *     JITTER
 *
 * so that all clients same time par
 * retry na karein.
 *
 *
 * ============================================================
 * 36. CIRCUIT BREAKER
 * ============================================================
 *
 * Circuit breaker:
 *
 *     Failing dependency ko repeatedly
 *     call karna stop karta hai.
 *
 *
 * Flow:
 *
 *     Dependency fails
 *          ↓
 *     Circuit opens
 *          ↓
 *     Requests fail fast / fallback
 *          ↓
 *     Dependency gets time to recover
 *
 *
 * Important:
 *
 *     Circuit breaker SPOF remove nahi karta.
 *
 *     It prevents one failing dependency
 *     from consuming the rest of the system.
 *
 *
 * ============================================================
 * 37. BACKPRESSURE
 * ============================================================
 *
 * Backpressure means:
 *
 *     Jab system saturated ho,
 *     incoming work ko control karo.
 *
 *
 * Useful mechanisms:
 *
 *     Queue limits
 *     Bulkheads
 *     Load shedding
 *     Rate limits
 *
 *
 * Goal:
 *
 *     Overload ko poore system mein
 *     spread hone se rokna.
 *
 *
 * ============================================================
 * 38. IDEMPOTENCY
 * ============================================================
 *
 * Retries ke saath important.
 *
 *
 * Problem:
 *
 *     Request succeeds
 *     Response lost
 *
 *     Client retries
 *
 *     Same write twice ho sakti hai.
 *
 *
 * Solution:
 *
 *     Idempotency key.
 *
 *
 * Example:
 *
 *     Payment
 *
 *     ID = abc123
 *
 *     First request:
 *
 *         Process payment
 *
 *     Retry:
 *
 *         abc123 already processed
 *
 *         Don't process again.
 *
 *
 * ============================================================
 * 39. MONITORING
 * ============================================================
 *
 * Monitoring SPOF remove nahi karta.
 *
 * But:
 *
 *     Failure detect karne ka time reduce karta hai.
 *
 *
 * Monitor:
 *
 *     Health-check failures
 *     Error rate
 *     Latency percentiles
 *     CPU
 *     Memory
 *     Disk
 *     Connections
 *     Thread pools
 *     Queue depth
 *     Replication lag
 *     Failover events
 *     Backup/restore status
 *     Dependency errors
 *     AZ/region traffic imbalance
 *
 *
 * ============================================================
 * 40. ALERTING
 * ============================================================
 *
 * Alert ka purpose:
 *
 *     Engineer ko actionable information dena.
 *
 *
 * Bad:
 *
 *     Alert fires
 *          ↓
 *     Nobody knows what to do.
 *
 *
 * This itself creates:
 *
 *     Operational SPOF
 *
 *
 * Good alert:
 *
 *     Problem
 *        +
 *     Clear owner
 *        +
 *     Clear action
 *
 *
 * ============================================================
 * 41. RUNBOOK
 * ============================================================
 *
 * Runbook = incident ke time
 * step-by-step recovery guide.
 *
 *
 * Runbook should explain:
 *
 *     1. Failure detect kaise karein?
 *
 *     2. Failover kaise karein?
 *
 *     3. Rollback kaise karein?
 *
 *     4. Recovery verify kaise karein?
 *
 *     5. Decision ka owner kaun hai?
 *
 *     6. Expected customer impact kya hai?
 *
 *
 * ============================================================
 * 42. REDUNDANCY HAS A COST
 * ============================================================
 *
 * SPOFs remove karna free nahi hai.
 *
 *
 * More redundancy means:
 *
 *     More infrastructure cost
 *     More operational complexity
 *     More moving parts
 *     More replication decisions
 *     More consistency issues
 *     More failover states
 *     More testing
 *     More configuration
 *
 *
 * Therefore:
 *
 *     "Make everything redundant"
 *
 * is NOT the correct system-design answer.
 *
 *
 * ============================================================
 * 43. WHEN CAN WE ACCEPT A SPOF?
 * ============================================================
 *
 * SPOF hamesha remove karna necessary nahi.
 *
 *
 * Acceptable examples:
 *
 *     Low-risk internal tool
 *     Early prototype
 *     Non-critical feature
 *
 *
 * But decision conscious hona chahiye.
 *
 *
 * For every SPOF decide:
 *
 *     1. Remove it now
 *
 *     2. Mitigate using graceful degradation
 *
 *     3. Monitor and accept the risk
 *
 *     4. Put it on roadmap
 *
 *
 * ============================================================
 * 44. BUSINESS IMPACT MATTERS
 * ============================================================
 *
 * Har component ko same engineering effort
 * dena necessary nahi.
 *
 *
 * Example:
 *
 *     Checkout Database
 *
 * vs
 *
 *     Badge-count Cache
 *
 *
 * Checkout DB:
 *
 *     Critical
 *     High investment justified.
 *
 *
 * Badge cache:
 *
 *     Less critical
 *     Temporary SPOF may be acceptable.
 *
 *
 * Therefore:
 *
 *     SPOF priority =
 *
 *     Business Impact
 *     +
 *     User Impact
 *     +
 *     Failure Probability
 *
 *
 * ============================================================
 * 45. SPOF ANALYSIS — INTERVIEW PROCESS
 * ============================================================
 *
 * System design interview mein agar interviewer
 * puche:
 *
 *     "How would you remove SPOFs?"
 *
 *
 * Answer step-by-step:
 *
 *
 * Step 1:
 *
 *     Identify critical user flows.
 *
 *
 * Step 2:
 *
 *     Map all dependencies.
 *
 *
 * Step 3:
 *
 *     Identify single dependencies.
 *
 *
 * Step 4:
 *
 *     Check whether alternatives exist.
 *
 *
 * Step 5:
 *
 *     Check shared failure domains.
 *
 *
 * Step 6:
 *
 *     Add redundancy where business impact
 *     justifies it.
 *
 *
 * Step 7:
 *
 *     Add failover.
 *
 *
 * Step 8:
 *
 *     Test failover.
 *
 *
 * Step 9:
 *
 *     Add graceful degradation.
 *
 *
 * Step 10:
 *
 *     Add monitoring + alerting + runbooks.
 *
 *
 * ============================================================
 * 46. SPOF vs REDUNDANCY
 * ============================================================
 *
 *
 * SPOF:
 *
 *     One critical component
 *          ↓
 *     Failure
 *          ↓
 *     System/critical flow fails
 *
 *
 * Redundancy:
 *
 *     Component A
 *       +
 *     Component B
 *
 *          ↓
 *
 *     A fails
 *          ↓
 *     B continues
 *
 *
 * BUT:
 *
 *     A + B same failure domain
 *
 *          ↓
 *
 *     Both may fail together
 *
 *
 * Therefore:
 *
 *     Redundancy must be independent.
 *
 *
 * ============================================================
 * 47. MOST IMPORTANT INTERVIEW QUESTIONS
 * ============================================================
 *
 *
 * Q1. What is a SPOF?
 *
 * Answer:
 *
 *     A single dependency/component/process whose failure
 *     can bring down the system or a critical user flow,
 *     when there is no working alternative.
 *
 *
 * ------------------------------------------------------------
 *
 * Q2. Is every failed component a SPOF?
 *
 * Answer:
 *
 *     No.
 *
 *     SPOF depends on:
 *
 *         Critical path
 *         No alternative
 *         Unacceptable impact
 *
 *
 *     An optional recommendation service may fail
 *     without affecting checkout.
 *
 *
 * ------------------------------------------------------------
 *
 * Q3. Can a load balancer be a SPOF?
 *
 * Answer:
 *
 *     Yes.
 *
 *     If only one load balancer exists and it fails,
 *     users cannot reach healthy application servers.
 *
 *
 * ------------------------------------------------------------
 *
 * Q4. Are two application servers enough
 *     to remove SPOF?
 *
 * Answer:
 *
 *     Not necessarily.
 *
 *     They might share:
 *
 *         Same AZ
 *         Same network
 *         Same database
 *         Same DNS
 *         Same control plane
 *
 *     They can also lack enough capacity
 *     to handle traffic after one fails.
 *
 *
 * ------------------------------------------------------------
 *
 * Q5. What is shared fate?
 *
 * Answer:
 *
 *     Jab supposedly redundant components
 *     same underlying dependency/failure domain
 *     share karte hain and together fail kar sakte hain.
 *
 *
 *     Example:
 *
 *         Primary + Replica
 *         same AZ
 *
 *     AZ fails -> both fail.
 *
 *
 * ------------------------------------------------------------
 *
 * Q6. What is the difference between
 *     replication and backup?
 *
 * Answer:
 *
 *     Replication maintains another copy for
 *     availability/failover.
 *
 *     Backup protects against:
 *
 *         accidental deletion
 *         corruption
 *         bad code
 *         ransomware
 *         operator mistakes
 *
 *     Replication alone is not backup.
 *
 *
 * ------------------------------------------------------------
 *
 * Q7. Active-active vs Active-passive?
 *
 * Answer:
 *
 *     Active-active:
 *         Multiple components serve traffic simultaneously.
 *
 *         Better capacity and faster failover.
 *
 *
 *     Active-passive:
 *         One active component and one standby.
 *
 *         Simpler, but failover path must be tested.
 *
 *
 * ------------------------------------------------------------
 *
 * Q8. How do you find SPOFs in a system?
 *
 * Answer:
 *
 *     Start with critical user flows.
 *     Then map every dependency.
 *
 *     For each dependency ask:
 *
 *         What happens if it fails?
 *         Is there a fallback?
 *         Is fallback capacity sufficient?
 *         Can components fail together?
 *         Can we recover?
 *
 *
 * ------------------------------------------------------------
 *
 * Q9. Why can a cache become a SPOF/cascading risk?
 *
 * Answer:
 *
 *     Cache failure may send all traffic to DB.
 *
 *
 *         Cache fails
 *             ↓
 *         DB traffic increases
 *             ↓
 *         DB overloads
 *             ↓
 *         DB fails
 *
 *
 *     Therefore fallback path must have
 *     enough capacity.
 *
 *
 * ------------------------------------------------------------
 *
 * Q10. Why is failover testing important?
 *
 * Answer:
 *
 *     Because an untested standby may fail
 *     when the first real incident occurs.
 *
 *     Architecture diagram only shows
 *     intended behavior.
 *
 *     Failure testing validates actual behavior.
 *
 *
 * ============================================================
 * 48. QUICK REVISION
 * ============================================================
 *
 *
 * SPOF
 *     -> Single critical dependency
 *        whose failure causes unacceptable impact.
 *
 *
 * Redundancy
 *     -> Backup components.
 *
 *
 * Active-Active
 *     -> Multiple components serve traffic.
 *
 *
 * Active-Passive
 *     -> Standby takes over after failure.
 *
 *
 * Shared Fate
 *     -> Redundant components can fail together.
 *
 *
 * Failover
 *     -> Switch to healthy backup.
 *
 *
 * Replication
 *     -> Maintain multiple data copies.
 *
 *
 * Backup
 *     -> Recover from corruption/deletion/etc.
 *
 *
 * Graceful Degradation
 *     -> Keep core functionality working.
 *
 *
 * Circuit Breaker
 *     -> Stop repeatedly calling failing dependency.
 *
 *
 * Backpressure
 *     -> Control work during overload.
 *
 *
 * Health Check
 *     -> Identify whether instance is actually ready.
 *
 *
 * Runbook
 *     -> Step-by-step incident recovery guide.
 *
 *
 * ============================================================
 * 49. GOLDEN RULE FOR SPOF
 * ============================================================
 *
 *
 * DON'T ASK:
 *
 *     "Do I have multiple servers?"
 *
 *
 * ASK:
 *
 *     "If this dependency fails right now,
 *      what stops working?"
 *
 *
 * Then ask:
 *
 *     "Who notices?"
 *
 *     "Is there a fallback?"
 *
 *     "Can the fallback handle the load?"
 *
 *     "Can the backup fail with the primary?"
 *
 *     "How do we recover?"
 *
 *
 * ============================================================
 * CORE SPOF MINDSET
 * ============================================================
 *
 *
 *       CRITICAL USER FLOW
 *               |
 *               v
 *         MAP DEPENDENCIES
 *               |
 *               v
 *       FIND SINGLE DEPENDENCY
 *               |
 *               v
 *        CHECK ALTERNATIVE
 *               |
 *          +----+----+
 *          |         |
 *         YES        NO
 *          |         |
 *          v         v
 *     Test it    Add redundancy
 *                    |
 *                    v
 *              Add failover
 *                    |
 *                    v
 *          Test failure scenario
 *                    |
 *                    v
 *          Add degradation/fallback
 *                    |
 *                    v
 *            Monitor + Runbook
 *
 *
 * ============================================================
 * FINAL INTERVIEW LINE
 * ============================================================
 *
 * "SPOF analysis is not just about finding
 * single servers. I start from critical user flows,
 * trace both runtime and recovery dependencies,
 * check for shared failure domains, and then decide
 * whether to remove, mitigate, monitor, or consciously
 * accept each SPOF based on business impact."
 *
 * ============================================================
 */

In [ ]:
/**
 * SINGLE POINT OF FAILURE (SPOF) — SYSTEM DESIGN
 *
 * Core Idea:
 *   SPOF = Single Point of Failure
 *   Aisa component/dependency/process jiske fail hone par poora system
 *   ya koi CRITICAL user flow fail ho jaye.
 *
 * IMPORTANT: SPOF sirf server nahi hota. Ho sakta hai:
 *   Database, Load Balancer, DNS, Cloud Region, Message Broker,
 *   Secrets Manager, Deployment Pipeline, Feature Flag Service,
 *   Network Path, Human/Expert
 *
 * Example:
 *   Users --> Load Balancer --> App1 / App2
 *   App1/App2 healthy hain BUT Load Balancer fails
 *   -> Users cannot reach App1/App2
 *   Therefore: Load Balancer = SPOF
 */

/**
 * 1. WHAT MAKES SOMETHING A SPOF?
 *    3 conditions important hain:
 *      1. Critical path depends on it
 *      2. Failure ke time koi working alternative nahi hai
 *      3. Failure ka impact unacceptable hai
 *
 *    Formula:
 *      Critical Dependency + No Working Alternative + High Impact = SPOF
 */

/**
 * 2. NOT EVERY FAILURE IS A SPOF
 *    Har failed component SPOF nahi hota.
 *
 *    Example: Recommendation Service FAILS
 *      Homepage popular products dikha sakti hai.
 *      Checkout -> still works | Homepage -> still works
 *
 *    Recommendation service:
 *      SPOF for personalized recommendations
 *      BUT NOT SPOF for the entire product.
 */

/**
 * 3. SPOF IS ABOUT USER FLOWS
 *    Focus "Kitne servers hain?" nahi.
 *    Focus: "Agar ye dependency fail hui, user ka kya impact hoga?"
 *
 *    SPOF analysis = User Flows + Dependencies + Failure Impact
 */

/**
 * 4. EXAMPLE ARCHITECTURE
 *
 *      Clients
 *         |
 *         v
 *      Load Balancer (SPOF)
 *         |
 *       /   \
 *      v     v
 *    App1   App2
 *      \     /
 *       Cache (Cascading Risk)
 *         |
 *         v
 *      Database (SPOF)
 *
 *    Load Balancer fails: clients can't reach app servers.
 *    Database fails     : reads/writes needing durable data stop.
 *    Cache fails        : may NOT directly stop the app, BUT:
 *      Cache fails -> requests go to DB -> DB gets huge traffic
 *      -> DB overload -> DB fails
 *
 *    This is called CASCADING FAILURE.
 */

/**
 * 5. HIDDEN SPOFs
 *    Architecture diagram mein obvious nahi hote.
 *
 *    Example: App Server 1 + App Server 2 look redundant,
 *    but both depend on One NAT Gateway.
 *    NAT Gateway fails -> App1 and App2 both lose network.
 *    Therefore: NAT Gateway = Hidden SPOF.
 *
 *    Other hidden SPOFs: one firewall, one route, one private network link,
 *    one DNS, one control-plane service.
 */

/**
 * 6. REDUNDANCY ON PAPER vs REAL REDUNDANCY
 *    Two app servers, each can handle 50% traffic; total traffic 100%.
 *    Server 1 fails -> Server 2 gets 100% but can handle only 50%
 *    -> Server 2 overloads -> system fails.
 *
 *    Two servers != automatically redundant.
 *
 *    Real redundancy requires: capacity, health checks, proper routing,
 *    failure isolation.
 */

/**
 * 7. COMMON SPOFs
 *
 *    TRAFFIC ENTRY : One Load Balancer / Ingress / DNS provider
 *                    -> Users can't reach service.
 *    COMPUTE       : One app instance / one Availability Zone
 *                    -> Requests fail when it goes down.
 *    DATA          : One database primary without failover
 *                    -> Reads/writes stop.
 *    CACHE         : One cache cluster without proper fallback
 *                    -> Database overload after cache failure.
 *    MESSAGING     : One message broker/queue/stream
 *                    -> Producers block, consumers stop.
 *    CONFIGURATION : One config service / feature flag service
 *                    -> Services can't safely start/update.
 *    SECRETS       : One secrets manager / token issuer
 *                    -> Authentication/startup failure.
 *    STORAGE       : One bucket / volume / filesystem
 *                    -> Files become unavailable.
 *    NETWORK       : One NAT gateway / route table / firewall / VPN / private link
 *                    -> Large part of system loses connectivity.
 *    OPERATIONS    : One deployment pipeline / runbook / cloud account / expert
 *                    -> Recovery depends on one process/person.
 */

/**
 * 8. OPERATIONAL SPOF
 *    SPOF sirf runtime architecture mein nahi hota.
 *    Incident ke time operational dependencies bhi SPOF ban sakti hain.
 *
 *    Example:
 *      Database has multiple replicas, BUT failover manually karna padta hai
 *      aur sirf ek engineer ko pata hai kaise.
 *      Engineer unavailable -> failover can't happen.
 *      Therefore: Human knowledge = SPOF.
 *
 *    Another example:
 *      Backup exists -> restore permission -> one unavailable account.
 *      Backup hai, but restore nahi kar sakte.
 */

/**
 * 9. AI / ML SYSTEMS MEIN SPOF
 *    Examples: one vector database, model gateway, queue, embedding service,
 *    policy service, feature store, quota service.
 *
 *    Example:
 *      Application --> Model Gateway --> LLM
 *      Model healthy hai, but Model Gateway fails
 *      -> Application can't reach LLM.
 *      Therefore: Model Gateway = SPOF.
 */

/**
 * 10. FAIL-CLOSED DECISION
 *     Kuch services intentionally fail-closed kar sakti hain.
 *
 *     Example: Policy Service FAILS on a high-risk request.
 *     Instead of allowing the request: REJECT.
 *     Security reasons ke liye ye correct behavior ho sakta hai.
 *
 *     Important: fail-closed behavior intentional, documented aur understood
 *     hona chahiye. Accidental failure nahi hona chahiye.
 */

/**
 * 11. HOW TO IDENTIFY SPOFs
 *     SPOF analysis = Dependency Analysis
 *
 *     Step 1: Critical user flows identify karo.
 *       Login, Checkout, Payment, Search, Document Upload,
 *       Model Inference, Admin Permission Change, Incident Rollback
 *
 *     Step 2: Har flow ki dependencies list karo.
 *       Services, databases, caches, queues, secrets, DNS, network,
 *       third-party APIs, control-plane services
 *
 *     Step 3: Har dependency ke liye ask:
 *       "Agar ye fail ho gaya, user flow ka kya hoga?"
 */

/**
 * 12. RUNTIME vs RECOVERY DEPENDENCY
 *     System ke 2 dependency graphs hote hain:
 *       1. RUNTIME GRAPH  : normal traffic serve karne ke liye kya required hai?
 *       2. RECOVERY GRAPH : failure ke baad repair karne ke liye kya required hai?
 *
 *     RUNTIME EXAMPLE:
 *       Client -> API -> Database   (runtime dependency: Database)
 *
 *     RECOVERY EXAMPLE:
 *       Database has replicas, but failover ke liye required hain:
 *       Secrets Manager + Deployment Tool + DNS + Manual Permission
 *
 *       Secrets Manager down -> app may still serve traffic
 *       BUT restart/recovery impossible ho sakta hai.
 *
 *     Therefore: recovery dependency bhi SPOF ban sakti hai.
 */

/**
 * 13. SHARED FATE
 *     Do components tabhi truly redundant hain jab woh independently
 *     fail kar sakein.
 *
 *     Example: Primary DB + Replica DB, dono same Availability Zone mein.
 *     AZ fails -> Primary FAIL, Replica FAIL.
 *     Replica exists BUT real redundancy nahi hai.
 *
 *     This is SHARED FATE.
 */

/**
 * 14. SHARED FATE EXAMPLES
 *     1. Primary DB + Replica DB in same AZ -> AZ failure -> both fail.
 *     2. Load Balancer A + B both depend on one DNS
 *        -> DNS fails -> both unreachable.
 *     3. Two services use the same Database
 *        -> Database fails -> both services affected.
 *     4. Primary data + Backup in same cloud account
 *        -> account compromised/deleted -> data + backup both affected.
 *
 *     KEY IDEA: redundancy is useful only when failure modes are
 *     sufficiently independent.
 */

/**
 * 15. FAILURE REVIEW
 *     Har important component ke liye ask:
 *       1. What if it becomes slow?
 *       2. What if it returns errors?
 *       3. What if it returns bad data?
 *       4. What if it is unreachable from one zone/region?
 *       5. What if it recovers with stale data?
 *       6. What if failover works but capacity is reduced?
 *
 *     IMPORTANT: slow dependency is especially dangerous.
 *       Hard failure   : health check detects failure.
 *       Slow dependency: threads wait, connection pools fill, queues grow,
 *                        retries increase -> entire service can degrade.
 */

/**
 * 16. TEST YOUR SPOF ASSUMPTIONS
 *     Architecture diagram sirf intended behavior dikhata hai.
 *     Real reliability check ke liye: controlled failure testing karo.
 *
 *     Start small:
 *       Kill one app instance, disable one cache node,
 *       block database replica, rebalance queue consumers.
 *
 *     Then bigger tests:
 *       Simulate AZ failure, break DNS in test environment,
 *       test backup restore, failover to another region.
 */

/**
 * 17. CHAOS ENGINEERING
 *     Intentionally failures create karke system ka behavior test karna.
 *
 *     BUT random failure injection alone is not enough.
 *     Better: define hypothesis first.
 *
 *     Example:
 *       "If Cache Cluster A fails: Checkout latency < 300 ms
 *        AND Database CPU < 70%"
 *
 *     Then failure inject karo and check whether hypothesis true hai.
 */

/**
 * 18. STRATEGIES TO REDUCE SPOFs
 *       1. Redundancy              8. Timeouts
 *       2. Load balancing          9. Retries with jitter
 *       3. Health checks          10. Circuit breakers
 *       4. Data replication       11. Backpressure
 *       5. Backup                 12. Monitoring
 *       6. Failure-domain isolation 13. Runbooks
 *       7. Graceful degradation
 */

/**
 * 19. REDUNDANCY
 *     Backup component available to take over after failure.
 *
 *     Examples: multiple app instances, DB replicas, multiple cache nodes,
 *     multiple message brokers, multiple DNS nameservers, multiple AZs,
 *     multiple trained engineers.
 *
 *     ACTIVE-ACTIVE
 *       Multiple components simultaneously serve traffic.
 *         User --> LB --> S1 / S2
 *       Benefits: better capacity, faster failover.
 *
 *     ACTIVE-PASSIVE
 *       One component handles traffic, another waits as standby.
 *       Primary fails -> Standby becomes Primary.
 *       Benefit: simpler to reason about.
 *       Risk   : failover path may not work if never tested.
 *
 *     IMPORTANT: a standby that has never been tested cannot be fully trusted.
 */

/**
 * 20. LOAD BALANCING + HEALTH CHECKS
 *       Traffic --> Load Balancer --> S1 / S2 / S3
 *
 *     Health checks determine which instances are actually healthy.
 *
 *     IMPORTANT: health check should check READINESS,
 *     not just PROCESS IS ALIVE.
 *
 *     Example: process running, but database unavailable
 *       Alive = YES | Ready = NO
 *       Traffic should not be sent to it.
 */

/**
 * 21. REQUEST DRAINING
 *     Failed/unhealthy server ko immediately kill karne ke bajay,
 *     possible ho to existing requests complete hone do.
 *
 *     New requests    : do not send.
 *     Existing requests: drain.
 *
 *     This avoids unnecessary request failures.
 */

/**
 * 22. LOAD BALANCER ITSELF CAN BE SPOF
 *     Bad design:
 *       Clients --> LB --> S1 / S2
 *       LB fails -> everything becomes unreachable.
 *
 *     Better:
 *       Clients --> DNS / Anycast / VIP --> LB-A (S1/S2) / LB-B (S3)
 *       LB-A fails -> LB-B can continue serving traffic.
 */

/**
 * 23. DATA REPLICATION
 *     Data ki multiple copies maintain karna.
 *
 *       Primary DB --> Replica A / Replica B
 *
 *     Primary fails -> another replica can potentially take over.
 */

/**
 * 24. SYNCHRONOUS REPLICATION
 *     Primary --> Replica
 *     Commit tab complete hota hai jab replica/quorum acknowledge kare.
 *
 *     Advantages   : better freshness, less replication lag.
 *     Disadvantage : higher latency.
 */

/**
 * 25. ASYNCHRONOUS REPLICATION
 *     Client --> Primary --> ACK immediately --> Replica catches up later
 *
 *     Advantage    : lower latency.
 *     Disadvantage : failover ke time recent data lost ho sakta hai.
 */

/**
 * 26. MULTI-PRIMARY REPLICATION
 *     Multiple database nodes writes accept kar sakte hain.
 *
 *     Benefit : higher availability.
 *     Problem : write conflicts -> conflict resolution required.
 */

/**
 * 27. READ REPLICAS
 *     Read traffic distribute karne ke liye:
 *       Primary DB --(replicate)--> Read Replica A / Read Replica B
 *
 *     Benefits: better read capacity, additional failover option.
 *     Problem : replicas may lag behind primary.
 */

/**
 * 28. REPLICATION != BACKUP
 *     VERY IMPORTANT INTERVIEW QUESTION: "Is replication same as backup?"
 *     Answer: NO.
 *
 *     Replication: keeps another copy updated.
 *     Problem    : bad data bhi replicate ho sakta hai.
 *       Application bug -> deletes 1 million records
 *       -> replication -> delete spreads to replicas
 *
 *     Backup protects against: accidental deletion, bad application code,
 *     operator mistake, data corruption, ransomware.
 *
 *     Critical data: Replication + Backup
 */

/**
 * 29. BACKUP MUST BE TESTED
 *     Backup exists != Recovery guaranteed.
 *
 *     Test: restore time, restore correctness, restore permissions.
 *     Backup ko at least once end-to-end restore karke test karo.
 *     Otherwise backup recoverability = unproven.
 */

/**
 * 30. GEOGRAPHIC / FAILURE-DOMAIN ISOLATION
 *     Components ko independent failure domains mein distribute karo.
 *
 *     Possible domains: process, host, rack, Availability Zone,
 *     region, cloud account, cloud provider.
 *
 *     Bad    : Primary DB + Replica DB in the same AZ.
 *     Better : Primary DB -> AZ-A, Replica DB -> AZ-B.
 *     AZ-A failure -> replica in AZ-B survives.
 */

/**
 * 31. MULTI-AZ vs MULTI-REGION
 *     MULTI-AZ    : usually a good first step for high availability.
 *     MULTI-REGION: useful for regional disaster recovery, global low latency.
 *
 *     Multi-region introduces: replication lag, consistency trade-offs,
 *     failover complexity, higher operational cost.
 *
 *     IMPORTANT: don't use multi-region just because "more redundancy is better."
 *     Business requirement dekho.
 */

/**
 * 32. GRACEFUL DEGRADATION
 *     Goal: optional dependency fail ho, but core product continues.
 *
 *     Example: Recommendation Service FAILS -> show trending products.
 *     Core transaction: STILL WORKS.
 */

/**
 * 33. GRACEFUL DEGRADATION EXAMPLES
 *     Recommendations fail   : Personalized -> Trending items
 *     Search degraded        : Full search -> Exact title matching -> Cached results
 *     Analytics delayed      : Show last-updated timestamp
 *     Third-party enrichment fails: Continue core transaction without enrichment
 *     Vector index rebuilding: Don't return confusing empty results;
 *                              show "Ingestion/index is being rebuilt."
 *
 *     IMPORTANT: explicit limited-mode response is better than silently
 *     returning stale/incomplete data.
 */

/**
 * 34. TIMEOUTS
 *     Remote service ko indefinitely wait nahi karna chahiye.
 *
 *     Example: Service A -> Service B, B slow hai.
 *     Without timeout: A keeps waiting -> threads occupied
 *     -> connection pool full -> A becomes unhealthy.
 *
 *     Every remote call should have an appropriate timeout.
 */

/**
 * 35. RETRIES + JITTER
 *     Temporary failure mein retry useful ho sakta hai.
 *
 *     BUT too many retries:
 *       Failure -> Retry -> More traffic -> More failure
 *
 *     Use bounded retries.
 *     Add JITTER so that all clients same time par retry na karein.
 */

/**
 * 36. CIRCUIT BREAKER
 *     Failing dependency ko repeatedly call karna stop karta hai.
 *
 *     Dependency fails -> circuit opens -> requests fail fast / fallback
 *     -> dependency gets time to recover
 *
 *     Important: circuit breaker SPOF remove nahi karta.
 *     It prevents one failing dependency from consuming the rest of the system.
 */

/**
 * 37. BACKPRESSURE
 *     Jab system saturated ho, incoming work ko control karo.
 *
 *     Mechanisms: queue limits, bulkheads, load shedding, rate limits.
 *     Goal: overload ko poore system mein spread hone se rokna.
 */

/**
 * 38. IDEMPOTENCY
 *     Retries ke saath important.
 *
 *     Problem: request succeeds, response lost, client retries
 *     -> same write twice ho sakti hai.
 *
 *     Solution: idempotency key.
 *     Example: Payment ID = abc123
 *       First request: process payment.
 *       Retry: abc123 already processed -> don't process again.
 */

/**
 * 39. MONITORING
 *     Monitoring SPOF remove nahi karta, but failure detect karne ka
 *     time reduce karta hai.
 *
 *     Monitor: health-check failures, error rate, latency percentiles,
 *     CPU, memory, disk, connections, thread pools, queue depth,
 *     replication lag, failover events, backup/restore status,
 *     dependency errors, AZ/region traffic imbalance.
 */

/**
 * 40. ALERTING
 *     Purpose: engineer ko actionable information dena.
 *
 *     Bad : alert fires -> nobody knows what to do
 *           (this itself creates an Operational SPOF).
 *     Good: Problem + Clear owner + Clear action.
 */

/**
 * 41. RUNBOOK
 *     Incident ke time step-by-step recovery guide.
 *
 *     Should explain:
 *       1. Failure detect kaise karein?
 *       2. Failover kaise karein?
 *       3. Rollback kaise karein?
 *       4. Recovery verify kaise karein?
 *       5. Decision ka owner kaun hai?
 *       6. Expected customer impact kya hai?
 */

/**
 * 42. REDUNDANCY HAS A COST
 *     SPOFs remove karna free nahi hai.
 *
 *     More redundancy = more infrastructure cost, operational complexity,
 *     moving parts, replication decisions, consistency issues,
 *     failover states, testing, configuration.
 *
 *     "Make everything redundant" is NOT the correct system-design answer.
 */

/**
 * 43. WHEN CAN WE ACCEPT A SPOF?
 *     SPOF hamesha remove karna necessary nahi.
 *
 *     Acceptable examples: low-risk internal tool, early prototype,
 *     non-critical feature.
 *
 *     But decision conscious hona chahiye. For every SPOF decide:
 *       1. Remove it now
 *       2. Mitigate using graceful degradation
 *       3. Monitor and accept the risk
 *       4. Put it on roadmap
 */

/**
 * 44. BUSINESS IMPACT MATTERS
 *     Har component ko same engineering effort dena necessary nahi.
 *
 *     Checkout Database: critical, high investment justified.
 *     Badge-count Cache: less critical, temporary SPOF may be acceptable.
 *
 *     SPOF priority = Business Impact + User Impact + Failure Probability
 */

/**
 * 45. SPOF ANALYSIS — INTERVIEW PROCESS
 *     Agar interviewer puche "How would you remove SPOFs?", answer step-by-step:
 *       1.  Identify critical user flows.
 *       2.  Map all dependencies.
 *       3.  Identify single dependencies.
 *       4.  Check whether alternatives exist.
 *       5.  Check shared failure domains.
 *       6.  Add redundancy where business impact justifies it.
 *       7.  Add failover.
 *       8.  Test failover.
 *       9.  Add graceful degradation.
 *       10. Add monitoring + alerting + runbooks.
 */

/**
 * 46. SPOF vs REDUNDANCY
 *     SPOF      : one critical component -> failure -> system/critical flow fails.
 *     Redundancy: Component A + Component B -> A fails -> B continues.
 *
 *     BUT if A + B are in the same failure domain, both may fail together.
 *     Therefore redundancy must be independent.
 */

/**
 * 47. MOST IMPORTANT INTERVIEW QUESTIONS
 *
 * Q1. What is a SPOF?
 * A : A single dependency/component/process whose failure can bring down the
 *     system or a critical user flow, when there is no working alternative.
 *
 * Q2. Is every failed component a SPOF?
 * A : No. SPOF depends on: critical path, no alternative, unacceptable impact.
 *     An optional recommendation service may fail without affecting checkout.
 *
 * Q3. Can a load balancer be a SPOF?
 * A : Yes. If only one load balancer exists and it fails, users can't reach
 *     healthy application servers.
 *
 * Q4. Are two application servers enough to remove SPOF?
 * A : Not necessarily. They might share the same AZ, network, database, DNS
 *     or control plane. They can also lack enough capacity to handle traffic
 *     after one fails.
 *
 * Q5. What is shared fate?
 * A : Jab supposedly redundant components same underlying dependency/failure
 *     domain share karte hain aur together fail kar sakte hain.
 *     Example: Primary + Replica in same AZ -> AZ fails -> both fail.
 *
 * Q6. Difference between replication and backup?
 * A : Replication maintains another copy for availability/failover.
 *     Backup protects against accidental deletion, corruption, bad code,
 *     ransomware, operator mistakes. Replication alone is not backup.
 *
 * Q7. Active-active vs Active-passive?
 * A : Active-active: multiple components serve traffic simultaneously;
 *     better capacity and faster failover.
 *     Active-passive: one active + one standby; simpler, but failover path
 *     must be tested.
 *
 * Q8. How do you find SPOFs in a system?
 * A : Start with critical user flows, map every dependency. For each ask:
 *     What happens if it fails? Is there a fallback? Is fallback capacity
 *     sufficient? Can components fail together? Can we recover?
 *
 * Q9. Why can a cache become a SPOF/cascading risk?
 * A : Cache failure may send all traffic to DB:
 *     Cache fails -> DB traffic increases -> DB overloads -> DB fails.
 *     So the fallback path must have enough capacity.
 *
 * Q10. Why is failover testing important?
 * A : An untested standby may fail when the first real incident occurs.
 *     Architecture diagram only shows intended behavior;
 *     failure testing validates actual behavior.
 */

/**
 * 48. QUICK REVISION
 *
 *   SPOF            -> Single critical dependency whose failure causes unacceptable impact.
 *   Redundancy      -> Backup components.
 *   Active-Active   -> Multiple components serve traffic.
 *   Active-Passive  -> Standby takes over after failure.
 *   Shared Fate     -> Redundant components can fail together.
 *   Failover        -> Switch to healthy backup.
 *   Replication     -> Maintain multiple data copies.
 *   Backup          -> Recover from corruption/deletion/etc.
 *   Graceful Degradation -> Keep core functionality working.
 *   Circuit Breaker -> Stop repeatedly calling failing dependency.
 *   Backpressure    -> Control work during overload.
 *   Health Check    -> Identify whether instance is actually ready.
 *   Runbook         -> Step-by-step incident recovery guide.
 */

/**
 * 49. GOLDEN RULE FOR SPOF
 *
 *   DON'T ASK: "Do I have multiple servers?"
 *   ASK      : "If this dependency fails right now, what stops working?"
 *
 *   Then ask:
 *     "Who notices?"
 *     "Is there a fallback?"
 *     "Can the fallback handle the load?"
 *     "Can the backup fail with the primary?"
 *     "How do we recover?"
 */

/**
 * CORE SPOF MINDSET
 *
 *       CRITICAL USER FLOW
 *               |
 *         MAP DEPENDENCIES
 *               |
 *       FIND SINGLE DEPENDENCY
 *               |
 *        CHECK ALTERNATIVE
 *          /          \
 *        YES           NO
 *         |             |
 *      Test it     Add redundancy
 *                       |
 *                  Add failover
 *                       |
 *              Test failure scenario
 *                       |
 *            Add degradation/fallback
 *                       |
 *               Monitor + Runbook
 */

/**
 * FINAL INTERVIEW LINE
 *   "SPOF analysis is not just about finding single servers. I start from
 *    critical user flows, trace both runtime and recovery dependencies,
 *    check for shared failure domains, and then decide whether to remove,
 *    mitigate, monitor, or consciously accept each SPOF based on
 *    business impact."
 */